# Sweep Analysis: f2fm6a3c

Analysis of the continual learning sweep comparing BP vs EFC across different task settings.

In [1]:
import os
import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
SWEEP_ID = "f2fm6a3c"
WANDB_DIR = Path("./wandb")
SWEEP_DIR = WANDB_DIR / f"sweep-{SWEEP_ID}"

In [3]:
def load_sweep_results(sweep_dir: Path, wandb_dir: Path) -> pd.DataFrame:
    """Load all run results from the local wandb logs for a given sweep."""
    results = []
    
    # Get all run IDs from sweep config files
    for config_file in sweep_dir.glob("config-*.yaml"):
        run_id = config_file.stem.replace("config-", "")
        
        # Find the corresponding run directory
        run_dirs = list(wandb_dir.glob(f"run-*-{run_id}"))
        if not run_dirs:
            print(f"Warning: No run directory found for {run_id}")
            continue
        
        run_dir = run_dirs[0]
        
        # Load config from sweep directory
        with open(config_file) as f:
            config = yaml.safe_load(f)
        
        # Load summary from run directory
        summary_file = run_dir / "files" / "wandb-summary.json"
        if not summary_file.exists():
            print(f"Warning: No summary file found for {run_id}")
            continue
            
        with open(summary_file) as f:
            summary = json.load(f)
        
        # Extract relevant info
        result = {
            "run_id": run_id,
            "setting": config.get("setting", {}).get("value"),
            "method": config.get("method", {}).get("value"),
            "seed": config.get("seed", {}).get("value"),
            "final_avg_accuracy": summary.get("final_avg_accuracy"),
            "final_forgetting": summary.get("final_forgetting"),
        }
        results.append(result)
    
    return pd.DataFrame(results)

In [4]:
# Load all results
df = load_sweep_results(SWEEP_DIR, WANDB_DIR)
print(f"Loaded {len(df)} runs")
df.head()

Loaded 60 runs


,run_id,setting,method,seed,final_avg_accuracy,final_forgetting
0,cwrrmkaz,TaskILMNIST,bp,42,60.500172,81.542900
1,vpg1g4r2,ClassILMNIST5Task,bp,42,3.966452,99.952719
2,7lyi5lw5,TaskILCIFAR10,bp,42,79.020000,33.750000
3,3j9xiw9q,ClassILCIFAR5Task,bp,42,4.036500,98.550000
4,hr1clm98,TaskILTinyImageNet,bp,42,8.880000,29.600000


In [5]:
# Check data completeness
print("Runs per setting and method:")
print(df.groupby(["setting", "method"]).size().unstack(fill_value=0))

Runs per setting and method:
method                     bp  efc
setting                           
ClassILCIFAR5Task           5    5
ClassILMNIST5Task           5    5
ClassILTinyImageNet10Task   5    5
TaskILCIFAR10               5    5
TaskILMNIST                 5    5
TaskILTinyImageNet          5    5


In [6]:
# Compute mean and std of final_avg_accuracy grouped by setting and method
accuracy_stats = df.groupby(["setting", "method"])["final_avg_accuracy"].agg(["mean", "std", "count"])
accuracy_stats = accuracy_stats.round(4)
accuracy_stats

mean     std  count
setting                   method                        
ClassILCIFAR5Task         bp       4.0170  0.0621      5
                          efc      6.5231  1.7418      5
ClassILMNIST5Task         bp       3.9838  0.0534      5
                          efc      6.7376  3.6836      5
ClassILTinyImageNet10Task bp       0.3962  0.0212      5
                          efc      0.4167  0.2185      5
TaskILCIFAR10             bp      79.4380  1.2687      5
                          efc     83.8120  1.0905      5
TaskILMNIST               bp      59.4125  0.8036      5
                          efc     65.3002  3.2414      5
TaskILTinyImageNet        bp       8.8960  0.2428      5
                          efc      8.9680  0.3906      5

In [7]:
# Create a clean results table: Setting vs Method with mean +/- std
# Note: final_avg_accuracy is already stored as percentage (0-100), not decimal (0-1)

# Pivot table with mean accuracy
pivot_mean = df.pivot_table(
    values="final_avg_accuracy", 
    index="setting", 
    columns="method", 
    aggfunc="mean"
)

# Pivot table with std
pivot_std = df.pivot_table(
    values="final_avg_accuracy", 
    index="setting", 
    columns="method", 
    aggfunc="std"
)

# Combine into formatted string (values are already percentages)
results_table = pivot_mean.copy()
for col in results_table.columns:
    results_table[col] = [
        f"{m:.2f}% +/- {s:.2f}%" 
        for m, s in zip(pivot_mean[col], pivot_std[col])
    ]

# Rename columns for display
results_table.columns = [col.upper() for col in results_table.columns]

# Order settings logically
setting_order = [
    "TaskILMNIST", "ClassILMNIST5Task",
    "TaskILCIFAR10", "ClassILCIFAR5Task",
    "TaskILTinyImageNet", "ClassILTinyImageNet10Task"
]
results_table = results_table.reindex(setting_order)

print("Final Average Accuracy (mean +/- std over 5 seeds)")
print("=" * 60)
results_table

Final Average Accuracy (mean +/- std over 5 seeds)


,BP,EFC
setting,,
TaskILMNIST,59.41% +/- 0.80%,65.30% +/- 3.24%
ClassILMNIST5Task,3.98% +/- 0.05%,6.74% +/- 3.68%
TaskILCIFAR10,79.44% +/- 1.27%,83.81% +/- 1.09%
ClassILCIFAR5Task,4.02% +/- 0.06%,6.52% +/- 1.74%
TaskILTinyImageNet,8.90% +/- 0.24%,8.97% +/- 0.39%
ClassILTinyImageNet10Task,0.40% +/- 0.02%,0.42% +/- 0.22%


In [ ]:
# Also create a numeric table for easier comparison
# Values are already percentages, no need to multiply by 100
numeric_table = pivot_mean.copy()
numeric_table.columns = [col.upper() for col in numeric_table.columns]
numeric_table = numeric_table.reindex(setting_order)
numeric_table = numeric_table.round(2)

print("Final Average Accuracy (%) - Numeric Values")
print("=" * 60)
numeric_table

In [ ]:
# Compute delta (EFC - BP) to see improvement
if "BP" in numeric_table.columns and "EFC" in numeric_table.columns:
    numeric_table["Delta (EFC - BP)"] = numeric_table["EFC"] - numeric_table["BP"]
    print("Accuracy Comparison with Delta")
    print("=" * 60)
    print(numeric_table)

In [ ]:
# Forgetting analysis
forgetting_table = df.pivot_table(
    values="final_forgetting", 
    index="setting", 
    columns="method", 
    aggfunc="mean"
)
forgetting_table.columns = [col.upper() for col in forgetting_table.columns]
forgetting_table = forgetting_table.reindex(setting_order)
forgetting_table = forgetting_table.round(2)

print("Final Forgetting (mean over 5 seeds) - Lower is better")
print("=" * 60)
forgetting_table